In [ ]:
import os
import json
import shutil
import random
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt

# Seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Check GPUs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f"Device: {device}")
print(f"Number of GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

if num_gpus < 2:
    print("\n⚠️ Only 1 GPU detected. Go to Settings → Accelerator → GPU T4 x2")

## 1. Find Dataset Paths

In [ ]:
# Explore /kaggle/input to find datasets
input_path = Path("/kaggle/input")
print("📁 Contents of /kaggle/input:")
for item in sorted(input_path.iterdir()):
    print(f"  {item.name}/")
    if item.is_dir():
        for sub in list(item.iterdir())[:5]:
            print(f"    {sub.name}")

In [ ]:
# Auto-detect dataset paths
def find_dataset_path(base_path, keywords, exclude_keywords=None):
    """Find dataset path by keywords"""
    exclude_keywords = exclude_keywords or []
    
    for item in base_path.iterdir():
        name_lower = item.name.lower()
        if any(kw in name_lower for kw in keywords):
            if any(ex in name_lower for ex in exclude_keywords):
                continue
            return item
    return None

def find_train_folder(dataset_path):
    """Recursively find train folder or folder with class subdirs"""
    if dataset_path is None:
        return None
    
    # Check common structures
    candidates = [
        dataset_path / "train",
        dataset_path / "Train",
    ]
    
    # Check nested structures
    for sub in dataset_path.iterdir():
        if sub.is_dir():
            candidates.append(sub / "train")
            candidates.append(sub / "Train")
            candidates.append(sub)
            for sub2 in sub.iterdir():
                if sub2.is_dir():
                    candidates.append(sub2 / "train")
                    candidates.append(sub2)
    
    for c in candidates:
        if c.exists() and c.is_dir():
            # Check if has class subdirs with images
            subdirs = [d for d in c.iterdir() if d.is_dir()]
            if subdirs:
                sample_dir = subdirs[0]
                imgs = list(sample_dir.glob("*.jpg")) + list(sample_dir.glob("*.JPG")) + list(sample_dir.glob("*.png"))
                if imgs:
                    return c
    
    return None

# Find datasets
ds1_base = find_dataset_path(input_path, ["new-plant-diseases", "new_plant_diseases"])
ds2_base = find_dataset_path(input_path, ["plant-disease", "plant_disease"], exclude_keywords=["new"])

DATASET1_TRAIN = find_train_folder(ds1_base)
DATASET2_PATH = find_train_folder(ds2_base)

print("="*60)
print("DETECTED PATHS:")
print("="*60)
print(f"Dataset 1 (vipoooool): {DATASET1_TRAIN}")
print(f"Dataset 2 (rashidthihan): {DATASET2_PATH}")

if DATASET1_TRAIN:
    print(f"\n✓ Dataset 1: {len(list(DATASET1_TRAIN.iterdir()))} class folders")
else:
    print("\n❌ Dataset 1 NOT FOUND!")

if DATASET2_PATH:
    print(f"✓ Dataset 2: {len(list(DATASET2_PATH.iterdir()))} class folders")
else:
    print("❌ Dataset 2 NOT FOUND!")

## 2. Analyze and Unify Class Names

In [ ]:
def normalize_class_name(name):
    """
    Normalize class name to unified format: Crop___Disease
    
    Handles:
    - "Crop___Disease" -> keep as is
    - "Disease (Crop)" -> "Crop___Disease"
    - "Disease_(Crop)" -> "Crop___Disease"
    """
    name = name.strip()
    
    # Already in Crop___Disease format
    if "___" in name:
        return name
    
    # Handle "Disease (Crop)" or "Disease_(Crop)" format
    # Match pattern: Something (Crop) or Something_(Crop)
    match = re.match(r'^(.+?)_?\((.+?)\)$', name)
    if match:
        disease = match.group(1).strip().replace(' ', '_')
        crop = match.group(2).strip().replace(' ', '_')
        
        # Handle special cases
        crop_mapping = {
            'Corn_(maize)': 'Corn_(maize)',
            'Corn_maize': 'Corn_(maize)',
            'Pepper,_bell': 'Pepper,_bell',
            'Pepper_bell': 'Pepper,_bell',
            'Cherry_(including_sour)': 'Cherry_(including_sour)',
            'Cherry_including_sour': 'Cherry_(including_sour)',
        }
        crop = crop_mapping.get(crop, crop)
        
        return f"{crop}___{disease}"
    
    # Handle special cases without parentheses
    if name == "Unknown_Disease" or name == "Unknown Disease":
        return "Unknown___Disease"
    
    # If no pattern matched, return with Unknown crop
    return f"Unknown___{name.replace(' ', '_')}"

# Test normalization
test_cases = [
    "Apple___Apple_scab",
    "BrownSpot (Rice)",
    "BrownSpot_(Rice)",
    "Anthracnose_(Mango)",
    "Target_spot_(Cotton)",
    "healthy (Grape)",
    "Cercospora_leaf_spot_Gray_leaf_spot_(Corn_(maize))",
]

print("Testing class name normalization:")
for tc in test_cases:
    print(f"  {tc} -> {normalize_class_name(tc)}")

In [ ]:
def collect_all_classes(dataset_path, source_name):
    """Collect all classes with image counts from a dataset"""
    classes = {}
    
    if dataset_path is None or not dataset_path.exists():
        print(f"❌ {source_name} path not found")
        return classes
    
    for class_dir in sorted(dataset_path.iterdir()):
        if not class_dir.is_dir():
            continue
        
        original_name = class_dir.name
        normalized_name = normalize_class_name(original_name)
        
        # Count images
        images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.JPG")) + \
                 list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpeg"))
        
        if images:
            classes[original_name] = {
                'normalized': normalized_name,
                'count': len(images),
                'path': class_dir,
                'source': source_name
            }
    
    return classes

# Collect from both datasets
ds1_classes = collect_all_classes(DATASET1_TRAIN, "Dataset1")
ds2_classes = collect_all_classes(DATASET2_PATH, "Dataset2")

print(f"\nDataset 1: {len(ds1_classes)} classes")
print(f"Dataset 2: {len(ds2_classes)} classes")

In [ ]:
# Find unified classes and handle duplicates
unified_classes = defaultdict(list)

# Add Dataset 1 classes
for orig, info in ds1_classes.items():
    unified_classes[info['normalized']].append(info)

# Add Dataset 2 classes
for orig, info in ds2_classes.items():
    unified_classes[info['normalized']].append(info)

# Analyze
print(f"\n{'='*60}")
print(f"UNIFIED CLASS ANALYSIS")
print(f"{'='*60}")
print(f"Total unified classes: {len(unified_classes)}")

# Find duplicates (classes with sources from both datasets)
duplicates = []
unique_ds1 = []
unique_ds2 = []

for norm_name, sources in unified_classes.items():
    ds1_present = any(s['source'] == 'Dataset1' for s in sources)
    ds2_present = any(s['source'] == 'Dataset2' for s in sources)
    
    if ds1_present and ds2_present:
        duplicates.append(norm_name)
    elif ds1_present:
        unique_ds1.append(norm_name)
    else:
        unique_ds2.append(norm_name)

print(f"\nClasses in both datasets (will merge): {len(duplicates)}")
print(f"Unique to Dataset 1: {len(unique_ds1)}")
print(f"Unique to Dataset 2 (new crops): {len(unique_ds2)}")

print(f"\n--- New crops from Dataset 2 ---")
for name in sorted(unique_ds2):
    count = sum(s['count'] for s in unified_classes[name])
    print(f"  {name}: {count} images")

## 3. Create Balanced Merged Dataset

In [ ]:
# Configuration
MAX_IMAGES_PER_CLASS = 2000  # Limit to prevent class imbalance
MIN_IMAGES_PER_CLASS = 50    # Minimum required
TRAIN_RATIO = 0.85

MERGED_PATH = Path("/kaggle/working/merged_dataset")
TRAIN_PATH = MERGED_PATH / "train"
VALID_PATH = MERGED_PATH / "valid"

# Clean up
if MERGED_PATH.exists():
    shutil.rmtree(MERGED_PATH)

MERGED_PATH.mkdir(parents=True)
TRAIN_PATH.mkdir()
VALID_PATH.mkdir()

print(f"Output: {MERGED_PATH}")
print(f"Max images per class: {MAX_IMAGES_PER_CLASS}")
print(f"Min images per class: {MIN_IMAGES_PER_CLASS}")
print(f"Train/Valid split: {TRAIN_RATIO:.0%}/{1-TRAIN_RATIO:.0%}")

In [ ]:
def copy_images_balanced(sources, target_train, target_valid, max_images, train_ratio):
    """
    Copy images from multiple sources with balancing
    """
    # Collect all image paths
    all_images = []
    for src in sources:
        src_path = src['path']
        images = list(src_path.glob("*.jpg")) + list(src_path.glob("*.JPG")) + \
                 list(src_path.glob("*.png")) + list(src_path.glob("*.jpeg"))
        all_images.extend(images)
    
    # Shuffle and limit
    random.shuffle(all_images)
    all_images = all_images[:max_images]
    
    # Split train/valid
    split_idx = int(len(all_images) * train_ratio)
    train_images = all_images[:split_idx]
    valid_images = all_images[split_idx:]
    
    # Create directories
    target_train.mkdir(exist_ok=True)
    target_valid.mkdir(exist_ok=True)
    
    # Copy train images
    for i, img in enumerate(train_images):
        dst = target_train / f"{i:05d}{img.suffix.lower()}"
        shutil.copy2(img, dst)
    
    # Copy valid images
    for i, img in enumerate(valid_images):
        dst = target_valid / f"{i:05d}{img.suffix.lower()}"
        shutil.copy2(img, dst)
    
    return len(train_images), len(valid_images)

# Process all classes
final_classes = []
skipped_classes = []
class_stats = []

print("\n" + "="*60)
print("CREATING MERGED DATASET")
print("="*60)

for norm_name in tqdm(sorted(unified_classes.keys()), desc="Processing classes"):
    sources = unified_classes[norm_name]
    total_images = sum(s['count'] for s in sources)
    
    # Skip classes with too few images
    if total_images < MIN_IMAGES_PER_CLASS:
        skipped_classes.append((norm_name, total_images))
        continue
    
    # Create class folders
    train_dir = TRAIN_PATH / norm_name
    valid_dir = VALID_PATH / norm_name
    
    # Copy with balancing
    n_train, n_valid = copy_images_balanced(
        sources, train_dir, valid_dir,
        MAX_IMAGES_PER_CLASS, TRAIN_RATIO
    )
    
    final_classes.append(norm_name)
    class_stats.append({
        'name': norm_name,
        'train': n_train,
        'valid': n_valid,
        'total_original': total_images,
        'sources': [s['source'] for s in sources]
    })

print(f"\n✓ Created {len(final_classes)} classes")
print(f"✗ Skipped {len(skipped_classes)} classes (too few images)")

if skipped_classes:
    print("\nSkipped classes:")
    for name, count in skipped_classes:
        print(f"  {name}: {count} images")

In [ ]:
# Save class names
final_classes = sorted(final_classes)
NUM_CLASSES = len(final_classes)

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(final_classes, f, indent=2)

print(f"\n{'='*60}")
print(f"FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total classes: {NUM_CLASSES}")
print(f"\nSaved to: /kaggle/working/class_names.json")

# Show class distribution
print(f"\n--- Class Distribution ---")
train_counts = [s['train'] for s in class_stats]
print(f"Train images per class: min={min(train_counts)}, max={max(train_counts)}, avg={np.mean(train_counts):.0f}")
print(f"Total train images: {sum(train_counts)}")
print(f"Total valid images: {sum(s['valid'] for s in class_stats)}")

# Show crop distribution
crops = Counter()
for name in final_classes:
    crop = name.split('___')[0]
    crops[crop] += 1

print(f"\n--- Crops Covered ---")
for crop, count in sorted(crops.items()):
    print(f"  {crop}: {count} diseases")

## 4. Create Data Loaders with Strong Augmentation

In [ ]:
class PlantDiseaseDataset(Dataset):
    def __init__(self, root_dir, class_names, transform=None):
        self.root_dir = Path(root_dir)
        self.class_names = class_names
        self.class_to_idx = {name: idx for idx, name in enumerate(class_names)}
        self.transform = transform
        self.samples = []
        
        for class_name in class_names:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                        self.samples.append((img_path, self.class_to_idx[class_name]))
        
        random.shuffle(self.samples)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            # Return a random valid sample on error
            return self.__getitem__(random.randint(0, len(self) - 1))

# Strong augmentation for training
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2)
])

# Validation transforms (no augmentation)
valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = PlantDiseaseDataset(TRAIN_PATH, final_classes, train_transforms)
valid_dataset = PlantDiseaseDataset(VALID_PATH, final_classes, valid_transforms)

print(f"Train samples: {len(train_dataset)}")
print(f"Valid samples: {len(valid_dataset)}")

In [ ]:
# Calculate class weights for balanced sampling
class_counts = Counter(label for _, label in train_dataset.samples)
total_samples = len(train_dataset)

# Weight = 1 / class_count (inversely proportional)
class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for _, label in train_dataset.samples]

# Create weighted sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# DataLoaders
BATCH_SIZE = 64 if num_gpus >= 2 else 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,  # Use weighted sampler for balanced training
    num_workers=4,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Using weighted sampling for class balance")

## 5. Build Model with Dropout for Regularization

In [ ]:
class PlantDiseaseModel(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        # Load pretrained EfficientNet-B0
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        
        # Replace classifier with custom head
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

# Create model
model = PlantDiseaseModel(NUM_CLASSES, dropout=0.4)

# Use DataParallel for multi-GPU
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel with {num_gpus} GPUs")

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Training with Early Stopping

In [ ]:
# Training config
EPOCHS = 20
LEARNING_RATE = 0.001
PATIENCE = 5  # Early stopping patience

# Loss and optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing helps
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=5, T_mult=2, eta_min=1e-6
)

print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Label smoothing: 0.1")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return total_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    # Per-class accuracy
    class_correct = defaultdict(int)
    class_total = defaultdict(int)
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Per-class stats
            for label, pred in zip(labels, predicted):
                class_total[label.item()] += 1
                if label == pred:
                    class_correct[label.item()] += 1
    
    # Find worst classes
    class_acc = {}
    for cls in class_total:
        class_acc[cls] = 100. * class_correct[cls] / class_total[cls]
    
    return total_loss / len(loader), 100. * correct / total, class_acc

In [ ]:
# Training loop with early stopping
best_acc = 0
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

print(f"\n{'='*60}")
print(f"STARTING TRAINING")
print(f"{'='*60}\n")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    valid_loss, valid_acc, class_acc = validate(model, valid_loader, criterion, device)
    
    # Update scheduler
    scheduler.step()
    
    # Log
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%")
    print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Show worst 5 classes
    sorted_acc = sorted(class_acc.items(), key=lambda x: x[1])
    print(f"Worst 5 classes:")
    for cls_idx, acc in sorted_acc[:5]:
        print(f"  {final_classes[cls_idx]}: {acc:.1f}%")
    
    # Save best model
    if valid_acc > best_acc:
        best_acc = valid_acc
        patience_counter = 0
        
        # Get model state (handle DataParallel)
        model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model_state,
            'class_names': final_classes,
            'num_classes': NUM_CLASSES,
            'valid_acc': valid_acc
        }, '/kaggle/working/disease_model_best.pth')
        
        print(f"✓ New best model saved! Acc: {valid_acc:.2f}%")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⚠️ Early stopping triggered at epoch {epoch+1}")
        break

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best Validation Accuracy: {best_acc:.2f}%")
print(f"{'='*60}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['valid_loss'], label='Valid Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['valid_acc'], label='Valid Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=150)
plt.show()

## 7. Test on Sample Images

In [ ]:
# Load best model for testing
checkpoint = torch.load('/kaggle/working/disease_model_best.pth')

test_model = PlantDiseaseModel(NUM_CLASSES)
test_model.load_state_dict(checkpoint['model_state_dict'])
test_model = test_model.to(device)
test_model.eval()

print(f"Loaded model with {checkpoint['num_classes']} classes")
print(f"Validation accuracy: {checkpoint['valid_acc']:.2f}%")

In [ ]:
# Test on random validation images
def predict_image(model, image_path, class_names, transform):
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        top5 = torch.topk(probs, 5)
    
    return [(class_names[idx], prob.item()*100) for idx, prob in zip(top5.indices, top5.values)]

# Get random samples from different crops
test_crops = ['Rice', 'Mango', 'Cotton', 'Grape', 'Tomato', 'Apple']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

sample_idx = 0
for crop in test_crops:
    # Find a class for this crop
    crop_classes = [c for c in final_classes if c.startswith(crop)]
    if not crop_classes:
        continue
    
    class_name = random.choice(crop_classes)
    class_dir = VALID_PATH / class_name
    
    if not class_dir.exists():
        continue
    
    images = list(class_dir.glob("*"))
    if not images:
        continue
    
    img_path = random.choice(images)
    predictions = predict_image(test_model, img_path, final_classes, valid_transforms)
    
    # Display
    img = Image.open(img_path)
    axes[sample_idx].imshow(img)
    axes[sample_idx].set_title(f"True: {class_name}\nPred: {predictions[0][0]}\n({predictions[0][1]:.1f}%)")
    axes[sample_idx].axis('off')
    
    sample_idx += 1
    if sample_idx >= 6:
        break

plt.tight_layout()
plt.savefig('/kaggle/working/sample_predictions.png', dpi=150)
plt.show()

## 8. Download Files

Download these files and place in your project's `backend/models/` folder:

1. `disease_model_best.pth` → rename to `disease_model_pytorch.pth`
2. `class_names.json` → keep same name

In [ ]:
# List output files
print("📦 Files to download:")
print("="*50)

output_files = [
    '/kaggle/working/disease_model_best.pth',
    '/kaggle/working/class_names.json',
    '/kaggle/working/training_history.png',
    '/kaggle/working/sample_predictions.png'
]

for f in output_files:
    if Path(f).exists():
        size = Path(f).stat().st_size / (1024*1024)
        print(f"✓ {Path(f).name} ({size:.1f} MB)")
    else:
        print(f"✗ {Path(f).name} (not found)")

print("\n" + "="*50)
print("Instructions:")
print("1. Click on the folder icon (📁) on the right")
print("2. Navigate to /kaggle/working/")
print("3. Right-click each file → Download")
print("4. Place in your project's backend/models/ folder")
print("5. Rename disease_model_best.pth to disease_model_pytorch.pth")